### Step 1: Data Generation and Visualization

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Generate synthetic sales data with trend, weekly/annual seasonality, and promotion effects
dates = pd.date_range("2021-01-01", periods=1095, freq="D")
t = np.arange(len(dates))
rng = np.random.default_rng(SEED)
promotion = (rng.random(len(t)) < 0.08).astype(float)
sales = (120 + 0.035*t + 18*np.sin(2*np.pi*t/7)
         + 10*np.sin(2*np.pi*t/365.25) + 25*promotion
         + rng.normal(0, 6, len(t)))

series = pd.DataFrame({"date": dates, "sales": sales.clip(min=0), "promotion": promotion})

display(series.head())
series.plot(x="date", y="sales", figsize=(12, 4), title="Daily sales")
plt.show()

### Step 2: Data Preprocessing and Windowing

In [ ]:
LOOKBACK = 30
split_index = int(len(series) * 0.80)
train_values = series[["sales"]].iloc[:split_index].to_numpy()
all_values = series[["sales"]].to_numpy()

# Scale data using MinMaxScaler fit only on training data to avoid leakage
scaler = MinMaxScaler()
scaler.fit(train_values)
scaled = scaler.transform(all_values).astype("float32")

def make_windows(values, lookback):
    X, y, indices = [], [], []
    for end in range(lookback, len(values)):
        X.append(values[end-lookback:end])
        y.append(values[end])
        indices.append(end)
    return np.array(X), np.array(y), np.array(indices)

X, y, target_indices = make_windows(scaled, LOOKBACK)
train_mask = target_indices < split_index

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]
test_dates = series["date"].iloc[target_indices[~train_mask]]

print("Training windows:", X_train.shape, "Test windows:", X_test.shape)

### Step 3: Model Architecture and Training

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input((LOOKBACK, 1)),
    tf.keras.layers.LSTM(48),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1),
], name="sales_lstm")

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])

history = model.fit(
    X_train, y_train, validation_split=0.15, epochs=50, batch_size=32,
    shuffle=False, verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True
    )],
)

model.summary()

### Step 4: Model Evaluation and Plotting Predictions

In [ ]:
pred_scaled = model.predict(X_test, verbose=0)
predictions = scaler.inverse_transform(pred_scaled).ravel()
actual = scaler.inverse_transform(y_test).ravel()

mae = mean_absolute_error(actual, predictions)
rmse = mean_squared_error(actual, predictions) ** 0.5
naive = scaler.inverse_transform(X_test[:, -1, :]).ravel()
naive_mae = mean_absolute_error(actual, naive)

print(f"LSTM MAE:  {mae:.2f}")
print(f"LSTM RMSE: {rmse:.2f}")
print(f"Naive previous-day MAE: {naive_mae:.2f}")

plt.figure(figsize=(13, 5))
plt.plot(test_dates, actual, label="Actual", linewidth=1.5)
plt.plot(test_dates, predictions, label="LSTM forecast", linewidth=1.2)
plt.xlabel("Date")
plt.ylabel("Sales")
plt.title("One-step-ahead forecast")
plt.legend()
plt.show()